In [ ]:
import xarray
import pathfinder2e_stats as pf2

%matplotlib inline

LEVEL = 4
HALF_HEALTH_ENEMY = False

In [ ]:
atk = pf2.tables.SIMPLE_PC.weapon_attack_bonus.exemplar.sum("component").sel(level=LEVEL).item()
atk

In [ ]:
weapon_dice = pf2.tables.PC.weapon_dice.striking_rune.sel(level=LEVEL).item()

immanence = 3 if HALF_HEALTH_ENEMY else 1
weapon = pf2.armory.picks.greatpick(weapon_dice).reduce_die() + pf2.Damage("spirit", 0, 0, immanence * weapon_dice)
weapon

In [ ]:
AC = pf2.tables.SIMPLE_NPC.AC.sel(level=LEVEL)
AC.display()

In [ ]:
off_guard = xarray.DataArray([0, -2], dims="off-guard", coords={"off-guard": [False, True]})
pf2.set_config(check_dependent_dims=["challenge", "off-guard"], damage_dependent_dims=["challenge", "off-guard"])

In [ ]:
atk_roll = pf2.check(atk, DC=AC + off_guard)
dmg_roll = pf2.damage(atk_roll, weapon)
barrows_edge_heal = dmg_roll.total_damage // 2
barrows_edge_heal.display()

In [ ]:
pf2.outcome_counts(atk_roll).display()

In [ ]:
total_heal = barrows_edge_heal.stack(col=["challenge", "off-guard"])
means = total_heal.mean("roll").to_pandas()
stds = total_heal.std("roll").to_pandas()
_ = means.plot.barh(xerr=stds)

In [ ]:
bins = barrows_edge_heal.max().item() + 1
_ = barrows_edge_heal.stack(col=["challenge", "off-guard"]).to_pandas().hist(bins=bins, figsize=(10, 10))